# Structured Output for VLMs
Our previous experiments have focused on open-ended image captioning. With this notebook, we begin to experiment with structured representations of images. For example, instead of saying, "This image shows a can of Diet Coca-Cola in a gold can," we would get:
- Object: can
- Product: soda
- Brand: coke
- Detail: gold can, caffine free

This notebook explores the following:
1. Can we use GPT-4 to develop an initial output from image captions in this structure? Using images as input? Using text as input?
2. Once (1) is validated by human annotators, can we use this to evaluate model accuracy and consistency at a finer-grain level?

## Import libraries


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

import copy
import json
import csv
import random
from datetime import datetime

import pandas as pd
import statistics as s
from scipy import stats

from tqdm import tqdm

import openai
from openai import OpenAI
OPENAI_CLIENT = OpenAI()
OPENAI_CLIENT.api_key = os.getenv("OPENAI_API_KEY")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

## Load Annotated Data

In [5]:
target_images_dtypes = {
    "image_id": int,
    "file_name": str,
    "vizwiz_url": str,
    "image_preview": str,
    "INCLUDE because product": str,
    "EXCLUDE because not verifable": str,
    "EXCLUDE because Book/DVD/CD/magazine?": str,
    "text_detected": str,
    "IQ quality check": str,
    "unrecognizable": bool,
    "framing": bool,
    "blur": bool,
    "obstruction": bool,
    "rotation": bool,
    "too dark": bool,
    "too bright": bool,
    "other": bool,
    "no issue": bool,
    "unrecognizable_orig": int,
    "framing_orig": int,
    "blur_orig": int,
    "obstruction_orig": int,
    "rotation_orig": int,
    "too dark_orig": int,
    "too bright_orig": int,
    "other": int,
    "no issue": int,
    "human_caption_0": str,
    "human_caption_1": str,
    "human_caption_2": str,
    "human_caption_3": str,
    "human_caption_4": str,
    "gpt-4o-2024-08-06_caption": str,
    "Llama-3.2-11B-Vision-Instruct_caption": str,
    "Molmo-7B-O-0924_caption": str,
    "general_notes": str,
    "gpt-4o-2024-08-06_notes": str,
    "Llama-3.2-11B-Vision-Instruct_notes": str,
    "Molmo-7B-O-0924_notes": str,
    "image_preview": str,
    "unable_to_verify": str,
    "gpt4o_code": str,
    "llama_code": str,
    "molmo_code": str,
    "notes": str,
    "double code notes": str,
    "double verified": str,
    "curved label": str,
    "text panel": str,
    "AMP_rotation": str,
    "XT_rotation": str,
}

In [15]:
# load previously annotated data, using dtypes above
annotated_df = pd.read_csv(
    "../study-analysis/study-2-product-analysis/annotated-data/final-annotated-images_1696-images_with-expert_2025-06-23_13-58.csv",
    dtype=target_images_dtypes,
    keep_default_na=False
)

# remove unable to verify data
annotated_df = annotated_df[annotated_df["unable_to_verify"] != "yes"]
print(f'Number of rows: {len(annotated_df)}')
annotated_df.head()

Number of rows: 1220


,image_id,file_name,vizwiz_url,image_preview,human_captions,annotator,notes,unable_to_verify,double code notes,double verified,gpt4o_caption,gpt4o_code,llama_caption,llama_code,molmo_caption,molmo_code,text_detected,unrecognizable_orig,framing_orig,blur_orig,obstruction_orig,rotation_orig,too_dark_orig,too_bright_orig,other_orig,curved label,text panel,AMP_rotation,XT_rotation,unrecognizable,framing,blur,obstruction,rotation,too dark,too bright,other,blur_framing,blur_rotation,framing_rotation,blur_framing_rotation,expert_caption
437,16588,VizWiz_train_00016588.jpg,https://vizwiz.cs.colorado.edu/VizWiz_visualiz...,,"House keys, medicine, and a package with the t...",Anne Marie,pall mall cigarettes,,,x,A pack of Pall Mall cigarettes with a green de...,yes,The image depicts a green Pall Mall cigarette ...,yes,A pack of Pall Mall cigarettes is visible on a...,yes,True,0,0,0,0,0,4,0,0,,,,,False,False,False,False,False,True,False,0,False,False,False,False,
438,15621,VizWiz_train_00015621.jpg,https://vizwiz.cs.colorado.edu/VizWiz_visualiz...,,a person's hand is covering a bottle of bubble...,Anne Marie,bath and body works bubble bath moonlight path,,,x,"A hand is holding a bottle labeled ""moonlight ...",yes++,The object is a plastic bottle of bubble bath ...,no,A hand with two gold rings is holding a rectan...,no,True,0,0,0,4,0,1,0,1,,,,,False,False,False,True,False,False,False,0,False,False,False,False,
439,7225,VizWiz_train_00007225.jpg,https://vizwiz.cs.colorado.edu/VizWiz_visualiz...,,A package of cigarettes resting on black leath...,Anne Marie,marlboro cigarettes,,,x,A pack of Marlboro cigarettes with a red and w...,yes,The image shows a pack of Marlboro cigarettes ...,yes,A pack of Marlboro cigarettes is visible on a ...,yes,True,0,0,0,5,0,0,0,0,,,,,False,False,False,True,False,False,False,0,False,False,False,False,
440,3429,VizWiz_train_00003429.jpg,https://vizwiz.cs.colorado.edu/VizWiz_visualiz...,,A 2 liter bottle of Pepsi and someone's hand.\...,Kapil,Bottle of regular pepsi,,,x,"A bottle of Pepsi with a red, white, and blue ...",yes,The image shows a dark brown glass bottle with...,yes,A Pepsi bottle is partially visible on a woode...,yes,True,0,0,0,4,0,0,0,0,x,,,,False,False,False,True,False,False,False,0,False,False,False,False,
441,17399,VizWiz_train_00017399.jpg,https://vizwiz.cs.colorado.edu/VizWiz_visualiz...,,A seasoned steak food product in white packagi...,Anne Marie,seasoned steak carmel color,,,x,"A packaging bag with visible text that reads ""...",yes,The image appears to be a close-up of a white ...,yes,The image shows a plastic bag containing a foo...,yes,True,0,1,0,0,0,4,0,0,,,,,False,False,False,False,False,True,False,0,False,False,False,False,


In [21]:
# create an object that's easily manipulable
relevant_columns = [
    "image_id",
    "file_name",
    "vizwiz_url",
    "image_preview",
    "human_captions",
    "notes",
    "gpt4o_caption",
    "gpt4o_code",
    "unrecognizable",
    "framing",
    "blur",
    "obstruction",
    "rotation",
    "too dark",
    "too bright",
    "other"
]
dataset_to_label_dict = annotated_df[relevant_columns].to_dict(orient='records')
dataset_to_label_dict[0]

{'image_id': 16588,
 'file_name': 'VizWiz_train_00016588.jpg',
 'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00016588.jpg',
 'image_preview': '',
 'human_captions': 'House keys, medicine, and a package with the text "Pall Mall"\nCar keys laying next to Pall Mall cigarettes\nSET OF KEYS AND A PACK OF CIGARETTES SITTING ON A COUNTER TOP\nA box of Pall Mall cigarettes is on a table.\nA dark picture of a pack of Pall Mall Green cigarettes.',
 'notes': 'pall mall cigarettes',
 'gpt4o_caption': 'A pack of Pall Mall cigarettes with a green design is next to a set of keys on a dark surface. The keys include a keychain with an attached tag.',
 'gpt4o_code': 'yes',
 'unrecognizable': False,
 'framing': False,
 'blur': False,
 'obstruction': False,
 'rotation': False,
 'too dark': True,
 'too bright': False,
 'other': 0}

## Part 1: Develop an Initial Hierarchy

### Create image based and text based prompts

In [61]:
def generate_structured_caption(image_url, openai_client, temperature=1.0):
    """
    Generates a caption for an image.

    Inputs:
    - image_url (str): url of image to caption.
    - prompt (str): prompt to use for captioning.
    - temperature (float; optional): temperature setting for model, greater than 0. Defaults to 1.0; lower values are more deterministic.

    Output:
    - (str): caption for image.
    """
    prompt = """
        You are a helpful assistant who describes objects in images for blind and low-vision individuals. Identify what the user is looking at in the image. Given a single image, output only a JSON object that validates against this schema:
        {
        "object": {"type": "string" or null, "confidence": 0.0–1.0},
        "product": {"type": "string" or null, "confidence": 0.0–1.0},
        "brand": {"type": "string" or null, "confidence": 0.0–1.0},
        "details": [
        {"type": "string", "confidence": 0.0–1.0},
        ...
        ]
        }
        
        Requirements:
        - All strings: lowercase, concise.
        - confidence: float [0.0,1.0], indicating your certainty.
        - Unknown fields → "value": null, "confidence": 0.0 (or empty details array).
        - Do not use vague adjectives like 'large' or 'small', and vague adverbs like 'prominently' or 'clearly'.
        - Do not mention camera artifacts (e.g., blur) or if an object is partially visible.
        - No extra keys or text; output only valid JSON.
        """
    
    # response = openai_client.responses.create(
    #     model="gpt-4o-2024-08-06",
    #     input=[
    #         {
    #             "role": "system",
    #             "content": [
    #                 {
    #                     "type": "input_text",
    #                     "text": prompt,
    #                 }
    #             ],
    #         },
    #         {
    #             "role": "user",
    #             "content": [
    #                 {
    #                     "type": "input_image",
    #                     "image_url": image_url,
    #                 }
    #             ],
    #         },
    #     ],
    #     text={"format": {"type": "text"}},
    #     reasoning={},
    #     tools=[],
    #     temperature=temperature,
    #     max_output_tokens=300,
    #     top_p=1,
    #     store=False,
    # )

    # # if response.output_text is not None:
    # #     return response.output_text
    # # else:
    # #     return ""
    # result_text = response.output_text
    # try:
    #     parsed = json.loads(result_text)
    # except json.JSONDecodeError:
    #     parsed = {"error": "Model returned invalid JSON", "raw": result_text}
    # return parsed

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-2024-08-06",
            messages=[
                {
                    "role": "system",
                    "content": prompt
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": image_url
                            }
                        }
                    ]
                }
            ],
            response_format={"type": "json_object"},
            temperature=temperature,
            max_tokens=300,
            top_p=1
        )
        
        result_text = response.choices[0].message.content
        
        if result_text is None:
            return {"error": "Model returned no content", "raw": None}
        
        try:
            parsed = json.loads(result_text)
            return parsed
        except json.JSONDecodeError as e:
            return {"error": f"Model returned invalid JSON: {str(e)}", "raw": result_text}
            
    except Exception as e:
        return {"error": f"API call failed: {str(e)}", "raw": None}

In [63]:
generate_structured_caption(dataset_to_label_dict[3]["vizwiz_url"], client)

{'object': {'type': 'plastic bottle', 'confidence': 1.0},
 'product': {'type': 'soda', 'confidence': 1.0},
 'brand': {'type': 'pepsi', 'confidence': 1.0},
 'details': [{'type': 'blue and red label', 'confidence': 0.9},
  {'type': '20 fl oz', 'confidence': 0.9},
  {'type': '591 ml', 'confidence': 0.9},
  {'type': '250 calories', 'confidence': 0.9}]}

### Apply to a sample of 100 images, with 75 correct and 50 incorrect.

In [65]:
dataset_to_label_dict[0]

{'image_id': 16588,
 'file_name': 'VizWiz_train_00016588.jpg',
 'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00016588.jpg',
 'image_preview': '',
 'human_captions': 'House keys, medicine, and a package with the text "Pall Mall"\nCar keys laying next to Pall Mall cigarettes\nSET OF KEYS AND A PACK OF CIGARETTES SITTING ON A COUNTER TOP\nA box of Pall Mall cigarettes is on a table.\nA dark picture of a pack of Pall Mall Green cigarettes.',
 'notes': 'pall mall cigarettes',
 'gpt4o_caption': 'A pack of Pall Mall cigarettes with a green design is next to a set of keys on a dark surface. The keys include a keychain with an attached tag.',
 'gpt4o_code': 'yes',
 'unrecognizable': False,
 'framing': False,
 'blur': False,
 'obstruction': False,
 'rotation': False,
 'too dark': True,
 'too bright': False,
 'other': 0}

In [81]:
correct_images = list(filter(lambda x: x["gpt4o_code"] == "yes", dataset_to_label_dict))
incorrect_images = list(filter(lambda x: x["gpt4o_code"] == "no", dataset_to_label_dict))

# select samples
correct_sample_size = 50
incorrect_sample_size = 25
correct_sample = random.sample(correct_images, correct_sample_size)
incorrect_sample = random.sample(incorrect_images, incorrect_sample_size)

In [82]:
for index, item in enumerate(tqdm(correct_sample)):
    correct_sample[index]["structured_caption"] = generate_structured_caption(item["vizwiz_url"], OPENAI_CLIENT)

for index, item in enumerate(tqdm(incorrect_sample)):
    incorrect_sample[index]["structured_caption"] = generate_structured_caption(item["vizwiz_url"], OPENAI_CLIENT)

100%|██████████| 25/25 [02:13<00:00,  5.35s/it]


In [83]:
incorrect_sample[0]

{'image_id': 7264,
 'file_name': 'VizWiz_train_00007264.jpg',
 'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00007264.jpg',
 'image_preview': '',
 'human_captions': 'A box of PureFit granola sits on top of a black surface, the box has a text description.\nA box of PureFit brand protein bars in an almond flavor.\na paper label of Pure Fit GRANOLA CRUNCH\nA container of Purefit Granola Crunch Purefit nutrition bars.\nPhoto is of a box of PureFit brand granola crunch.',
 'notes': 'purefit granola crunch cereal',
 'gpt4o_caption': 'A box of PureFit Granola Crunch Bars is shown upside down, with a close-up image of granola pieces and almonds on the front. The packaging features red and white colors with text that includes nutritional information and a "+6g Fiber" label.',
 'gpt4o_code': 'no',
 'unrecognizable': False,
 'framing': False,
 'blur': True,
 'obstruction': False,
 'rotation': True,
 'too dark': False,
 'too bright': False,
 'other': 0,
 'stru

In [103]:
def expand_into_keys(dataset):
    # expand the data into individual keys
    for index, item in enumerate(dataset):
        try:
            dataset[index]["object"] = item["structured_caption"]["object"]["type"]
            dataset[index]["product"] = item["structured_caption"]["product"]["type"]
            dataset[index]["brand"] = item["structured_caption"]["brand"]["type"]
            dataset[index]["details"] = "\n".join([detail["type"] for detail in item["structured_caption"]["details"]])
        except Exception as e:
            print(f'Error with {item["image_id"]}: {item["structured_caption"]}')


In [104]:
expand_into_keys(correct_sample)

Error with 10336: {'object': 'medicine box', 'confidence': 0.9, 'product': 'allergy medication', 'brand': 'benadryl', 'details': ['pink and blue packaging', "text with 'allergy'"]}


In [105]:
expand_into_keys(incorrect_sample)

In [110]:
# convert to pandas
output_df = pd.concat([pd.DataFrame.from_dict(correct_sample), pd.DataFrame.from_dict(incorrect_sample)])
output_df["KG Notes"] = ""
output_df["AMP Notes"] = ""
output_df.to_csv('./structured_output.csv', index=False)